# 04 — Mô hình 4: XLM-RoBERTa large (mô hình tham khảo bên ngoài)

Notebook này chạy **hai** cấu hình trên cùng một checkpoint bên ngoài:

1. **Zero-shot** `joeddav/xlm-roberta-large-xnli` — mô hình đã fine-tune trên XNLI
   (gồm cả tiếng Việt), đánh giá thẳng trên test set ViANLI, **không** train thêm.
   Cho biết một mô hình NLI đa ngữ mạnh sẵn có làm được gì trên bộ adversarial này.
2. **Fine-tune** `xlm-roberta-large` trên ViANLI — để so sánh công bằng với PhoBERT
   ở notebook 03 (cùng split, cùng metric, cùng tiêu chí chọn checkpoint).

## Thông tin nguồn (bắt buộc theo đề bài)

| Hạng mục | Giá trị |
|---|---|
| Tên mô hình | XLM-RoBERTa large |
| Nguồn công bố | Conneau et al., *Unsupervised Cross-lingual Representation Learning at Scale*, ACL 2020 |
| URL | https://huggingface.co/xlm-roberta-large — biến thể XNLI: https://huggingface.co/joeddav/xlm-roberta-large-xnli |
| Phiên bản/commit | điền commit hash thật vào biến `REVISION` bên dưới rồi chép vào báo cáo |
| Giấy phép | MIT |
| Mức độ sử dụng | chạy nguyên bản (zero-shot) **và** fine-tune trên ViANLI |
| Điều chỉnh của nhóm | thay classification head 3 lớp; ánh xạ lại thứ tự nhãn; bật fp16 + gradient checkpointing để vừa VRAM 16GB |

**Kiểm tra rò rỉ:** XNLI được xây từ MultiNLI (tiếng Anh, dịch sang 15 ngôn ngữ),
hoàn toàn tách biệt với ViANLI (do UIT-NLP xây riêng trên ngữ liệu tiếng Việt).
Vì vậy checkpoint XNLI **không** được huấn luyện trên test set của đề tài.
Vẫn nên tự xác nhận lại điều này khi đọc paper ViANLI và ghi vào báo cáo.

## Setup Kaggle — kéo code từ GitHub

Chạy cell này **trước tiên** trên Kaggle. Bỏ qua được khi chạy ở máy local.
Sửa code ở máy → `git push` → chạy lại cell này để lấy bản mới.

In [1]:
import os

REPO_URL = "https://github.com/dofu18/ViANLI_DL_NLP.git"

if os.path.exists("/kaggle/input"):
    !rm -rf /kaggle/working/repo
    !git clone -q $REPO_URL /kaggle/working/repo
    !cp -r /kaggle/working/repo/src /kaggle/working/
    !cp -r /kaggle/working/repo/configs /kaggle/working/
    !cp -r /kaggle/working/repo/data /kaggle/working/     # split cố định từ 01_eda
    print("src/:", sorted(os.listdir("/kaggle/working/src")))
else:
    print("Chạy local — bỏ qua bước clone.")

src/: ['data.py', 'evaluate.py', 'models.py', 'train.py', 'trainer.py']


In [2]:
!pip install -q "transformers>=4.44" "datasets>=2.21" accelerate sentencepiece

import json, os, sys, time

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

FIG_DIR = os.path.join(ROOT, "outputs", "figures")
LOG_DIR = os.path.join(ROOT, "outputs", "logs")
CKPT_DIR = os.path.join(ROOT, "outputs", "checkpoints")
for d in (FIG_DIR, LOG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

from data import LABELS, label_to_id, load_splits, normalize, set_seed
from models import count_params
from evaluate import (error_examples, full_report, hf_compute_metrics,
                      plot_confusion_matrix, plot_learning_curve)

SEED = 42
set_seed(SEED)
assert torch.cuda.is_available(), "Bật GPU trong Settings của Kaggle Notebook"
print(torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4 15.6 GB


## Chọn phần chạy (Part A / Part B)

In [3]:
PART = "A"   # "A" = zero-shot + fine-tune XLM-R large | "B" = ablation XNLI
#
# Chạy HAI Kaggle Version riêng: một lần PART="A", một lần PART="B".
# Lần chạy trước gộp cả hai vào một session, OOM ở run cuối và mất luôn
# xlmr_results.csv + toàn bộ predictions/ của những run đã thành công.

PRED_DIR = os.path.join(LOG_DIR, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)


def save_run(run_name):
    """Ghi summary + history + predictions của MỘT run ra đĩa ngay lập tức."""
    r = RESULTS[run_name]
    summary = {k: v for k, v in r.items() if not k.startswith("_")}
    with open(os.path.join(LOG_DIR, f"{run_name}_summary.json"), "w",
              encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    if r.get("_history"):
        with open(os.path.join(LOG_DIR, f"{run_name}_history.json"), "w",
                  encoding="utf-8") as f:
            json.dump(r["_history"], f, ensure_ascii=False, indent=2)
    y_true, y_pred = r["_preds"]
    np.save(os.path.join(PRED_DIR, f"{run_name}.npy"), np.asarray(y_pred))
    y_true_path = os.path.join(PRED_DIR, "y_true.npy")
    if os.path.exists(y_true_path):
        assert np.array_equal(np.load(y_true_path), np.asarray(y_true)), (
            "y_true lệch so với file đã lưu — thứ tự test set không nhất quán "
            "giữa các mô hình, mọi so sánh chéo ở nb 05 sẽ sai.")
    else:
        np.save(y_true_path, np.asarray(y_true))
    print(f"[save_run] {run_name} -> summary/history/predictions")

## 1. Nạp split cố định

Giống hệt notebook 02 và 03 — điều kiện bắt buộc để 4 mô hình so sánh được với nhau.

In [4]:
from datasets import Dataset, DatasetDict

SPLIT_DIR = os.path.join(ROOT, "data", "splits")
COLS = {"premise": "premise", "hypothesis": "hypothesis", "label": "label"}

if os.path.exists(os.path.join(SPLIT_DIR, "train.csv")):
    ds = DatasetDict({
        name: Dataset.from_pandas(
            pd.read_csv(os.path.join(SPLIT_DIR, f"{name}.csv"), encoding="utf-8")
              .fillna({"premise": "", "hypothesis": ""}))
        for name in ("train", "validation", "test")
    })
    cols = COLS
    print("Nạp từ data/splits/ (split cố định)")
else:
    ds, cols = load_splits(seed=SEED)
    print("CẢNH BÁO: chưa có data/splits/ — chạy 01_eda.ipynb trước")

y_test = np.array([label_to_id(y) for y in ds["test"][cols["label"]]])
{k: len(v) for k, v in ds.items()}

Nạp từ data/splits/ (split cố định)


{'train': 8012, 'validation': 1000, 'test': 1000}

# Phần A — Zero-shot XNLI

## 2. Nạp checkpoint và **ánh xạ lại thứ tự nhãn**

Đây là cái bẫy dễ mắc nhất: `joeddav/xlm-roberta-large-xnli` dùng thứ tự
`contradiction, neutral, entailment`, còn dự án này dùng
`entailment, neutral, contradiction`. Nếu không hoán vị, accuracy sẽ tệ một cách vô lý.
Đọc thẳng `config.id2label` thay vì hard-code.

In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ZS_MODEL_ID = "joeddav/xlm-roberta-large-xnli"
ZS_REVISION = "b227ee8435ceadfa86dc1368a34254e2838bf242"   # pin commit hash (yêu cầu của đề)
FT_REVISION = "c23d21b0620b635a76227c604d44e43a9f0ee389"   # xlm-roberta-large
MAX_LENGTH = 128     # chốt theo p95 của 01_eda (premise 51 + hypothesis 28 từ ~ 120 subword)

zs_tok = AutoTokenizer.from_pretrained(ZS_MODEL_ID, revision=ZS_REVISION)
zs_model = AutoModelForSequenceClassification.from_pretrained(
    ZS_MODEL_ID, revision=ZS_REVISION, torch_dtype=torch.float16).cuda().eval()

print("id2label của checkpoint:", zs_model.config.id2label)

# perm[i] = chỉ số logit của checkpoint tương ứng với LABELS[i] của dự án
src_label2id = {v.lower(): int(k) for k, v in zs_model.config.id2label.items()}
perm = [src_label2id[name] for name in LABELS]
print("Hoán vị áp dụng:", dict(zip(LABELS, perm)))

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


id2label của checkpoint: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
Hoán vị áp dụng: {'entailment': 2, 'neutral': 1, 'contradiction': 0}


In [6]:
@torch.no_grad()
def zero_shot_predict(split, batch_size=32):
    premises = [normalize(x) for x in split[cols["premise"]]]
    hypotheses = [normalize(x) for x in split[cols["hypothesis"]]]
    preds = []
    for i in range(0, len(premises), batch_size):
        batch = zs_tok(premises[i:i + batch_size], hypotheses[i:i + batch_size],
                       truncation=True, max_length=MAX_LENGTH,
                       padding=True, return_tensors="pt").to("cuda")
        logits = zs_model(**batch).logits.float().cpu().numpy()
        preds.append(logits[:, perm].argmax(-1))   # hoán vị về thứ tự của dự án
    return np.concatenate(preds)

t0 = time.time()
zs_pred = zero_shot_predict(ds["test"])
zs_seconds = time.time() - t0

RESULTS = {}
zs_report = full_report(y_test, zs_pred,
                        out_json=os.path.join(LOG_DIR, "xlmr_zeroshot_test.json"))
plot_confusion_matrix(y_test, zs_pred, "xlmr_zeroshot",
                      os.path.join(FIG_DIR, "xlmr_zeroshot_cm.png"))

RESULTS["xlmr_zeroshot"] = {
    "run_name": "xlmr_zeroshot", "model_id": ZS_MODEL_ID,
    "accuracy": zs_report["accuracy"], "macro_f1": zs_report["macro_f1"],
    "weighted_f1": zs_report["weighted_f1"],
    "params": count_params(zs_model), "epochs_chạy": 0, "train_seconds": 0.0,
    "inference_ms_per_sample": round(zs_seconds / len(y_test) * 1000, 2),
    "_preds": (y_test, zs_pred),
}
save_run("xlmr_zeroshot")

print(json.dumps({k: v for k, v in RESULTS["xlmr_zeroshot"].items()
                  if not k.startswith("_")}, ensure_ascii=False, indent=2))
print("\nPhân bố dự đoán:",
      {LABELS[i]: int((zs_pred == i).sum()) for i in range(len(LABELS))})

[save_run] xlmr_zeroshot -> summary/history/predictions
{
  "run_name": "xlmr_zeroshot",
  "model_id": "joeddav/xlm-roberta-large-xnli",
  "accuracy": 0.334,
  "macro_f1": 0.3246131097622273,
  "weighted_f1": 0.3246432136335971,
  "params": 559893507,
  "epochs_chạy": 0,
  "train_seconds": 0.0,
  "inference_ms_per_sample": 4.65
}

Phân bố dự đoán: {'entailment': 461, 'neutral': 407, 'contradiction': 132}


In [7]:
# Giải phóng VRAM trước khi fine-tune
del zs_model
torch.cuda.empty_cache()
print(f"VRAM đang dùng: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

VRAM đang dùng: 0.01 GB


# Phần B — Fine-tune XLM-R large trên ViANLI

## 3. Tokenize

XLM-R dùng SentencePiece nên **không** cần tách từ tiếng Việt (khác PhoBERT).

In [8]:
FT_MODEL_ID = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_ID, revision=FT_REVISION)

def tok(batch):
    out = tokenizer([normalize(x) for x in batch[cols["premise"]]],
                    [normalize(x) for x in batch[cols["hypothesis"]]],
                    truncation=True, max_length=MAX_LENGTH)
    out["labels"] = [label_to_id(y) for y in batch[cols["label"]]]
    return out

encoded = ds.map(tok, batched=True, remove_columns=ds["train"].column_names)
lens = [len(x) for x in encoded["train"]["input_ids"]]
print(f"độ dài subword: p50={np.percentile(lens, 50):.0f} "
      f"p95={np.percentile(lens, 95):.0f} max={max(lens)}")
print(f"tỉ lệ chạm trần {MAX_LENGTH}: {np.mean(np.array(lens) >= MAX_LENGTH):.2%}")

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8012 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

độ dài subword: p50=56 p95=92 max=128
tỉ lệ chạm trần 128: 0.57%


## 4. Fine-tune

**Lưu ý về XLM-R large:** mô hình này nổi tiếng dễ *collapse* — loss đứng yên và mô hình
dự đoán một nhãn duy nhất. Cách phòng: learning rate thấp (1e-5), warmup dài hơn,
gradient clipping. Nếu vẫn collapse, thử seed khác hoặc lr 5e-6 và **ghi lại hiện tượng
này trong báo cáo** — nó là một quan sát thực nghiệm có giá trị.

Cấu hình vừa VRAM 16GB: batch 8 × grad_accum 4 = effective batch 32 (bằng PhoBERT),
fp16 + gradient checkpointing.

In [9]:
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

def run(run_name, model_id=FT_MODEL_ID, learning_rate=1e-5, batch_size=8,
        grad_accum=4, epochs=4, patience=2, warmup_ratio=0.1):
    set_seed(SEED)
    revision = ZS_REVISION if model_id == ZS_MODEL_ID else FT_REVISION
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, revision=revision, num_labels=len(LABELS),
        ignore_mismatched_sizes=True)

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, run_name),
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,
        warmup_ratio=warmup_ratio,
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_strategy="epoch",
        logging_dir=os.path.join(LOG_DIR, run_name),
        seed=SEED,
        report_to=[],
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=encoded["train"], eval_dataset=encoded["validation"],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=hf_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
    )

    print(f"\n=== {run_name} | {count_params(model):,} tham số ===")
    started = time.time()
    trainer.train()
    elapsed = time.time() - started

    logs = trainer.state.log_history
    losses = {int(r["epoch"]): r["loss"] for r in logs if "loss" in r}
    history = [{"epoch": int(r["epoch"]),
                "train_loss": losses.get(int(r["epoch"]), float("nan")),
                "val_macro_f1": r["eval_macro_f1"],
                "val_accuracy": r["eval_accuracy"]}
               for r in logs if "eval_macro_f1" in r]

    t1 = time.time()
    pred = trainer.predict(encoded["test"])
    ms_per_sample = (time.time() - t1) / len(encoded["test"]) * 1000
    y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)

    report = full_report(y_true, y_pred,
                         out_json=os.path.join(LOG_DIR, f"{run_name}_test.json"))
    plot_learning_curve(history, run_name,
                        os.path.join(FIG_DIR, f"{run_name}_curve.png"))
    plot_confusion_matrix(y_true, y_pred, run_name,
                          os.path.join(FIG_DIR, f"{run_name}_cm.png"))

    summary = {"run_name": run_name, "model_id": model_id,
               "accuracy": report["accuracy"], "macro_f1": report["macro_f1"],
               "weighted_f1": report["weighted_f1"],
               "params": count_params(model), "epochs_chạy": len(history),
               "train_seconds": round(elapsed, 1),
               "inference_ms_per_sample": round(ms_per_sample, 2),
               "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
    RESULTS[run_name] = {**summary, "_preds": (y_true, y_pred),
                         "_history": history}
    # Ghi artifact NGAY: lần chạy trước chết ở run cuối và kéo theo mất cả
    # xlmr_results.csv lẫn thư mục predictions/ của những run đã xong.
    save_run(run_name)
    del trainer, model
    torch.cuda.empty_cache()
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("Phân bố dự đoán:",
          {LABELS[i]: int((y_pred == i).sum()) for i in range(len(LABELS))})
    return summary

In [10]:
if PART == "A":
    torch.cuda.reset_peak_memory_stats()
    # lr 1e-5 + warmup 0.06 làm epoch 1 collapse (macro-F1 0.166, grad_norm 179)
    # và contradiction recall chỉ còn 0.099. Hạ lr + kéo dài warmup.
    run("xlmr_large_finetuned", learning_rate=5e-6, warmup_ratio=0.1,
        epochs=5, patience=2)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is depreca


=== xlmr_large_finetuned | 559,893,507 tham số ===


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

[save_run] xlmr_large_finetuned -> summary/history/predictions
{
  "run_name": "xlmr_large_finetuned",
  "model_id": "xlm-roberta-large",
  "accuracy": 0.449,
  "macro_f1": 0.37209451746691097,
  "weighted_f1": 0.37230565198573934,
  "params": 559893507,
  "epochs_chạy": 5,
  "train_seconds": 2807.2,
  "inference_ms_per_sample": 16.23,
  "peak_vram_gb": 13.47
}
Phân bố dự đoán: {'entailment': 465, 'neutral': 514, 'contradiction': 21}


### Nếu mô hình collapse

Dấu hiệu: `val_macro_f1` xấp xỉ 0.17–0.20 (chỉ đoán một nhãn) và loss không giảm.
Chạy cell dưới để thử lại với learning rate thấp hơn — và **giữ lại cả hai kết quả**
để đưa vào phần thảo luận.

In [11]:
# (Nhánh COLLAPSED cũ đã bỏ: lr=5e-6 + warmup=0.1 nay là cấu hình mặc định
#  của xlmr_large_finetuned ở cell trên.)

## 5. Ablation: fine-tune từ checkpoint đã học XNLI

Khởi tạo từ `joeddav/xlm-roberta-large-xnli` thay vì `xlm-roberta-large` thuần —
kiểm tra xem kiến thức NLI từ XNLI có chuyển giao sang ViANLI hay không.
Đây chính là dạng **transfer learning** mà đề cương học phần yêu cầu vận dụng.

Lưu ý: head của checkpoint XNLI theo thứ tự nhãn khác, nhưng `ignore_mismatched_sizes=True`
sẽ khởi tạo lại head nên không cần hoán vị ở nhánh này.

In [12]:
if PART == "B":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # batch 4 + grad_accum 8 (effective 32) để không OOM như lần chạy trước
    run("xlmr_xnli_finetuned", model_id=ZS_MODEL_ID, epochs=3,
        batch_size=4, grad_accum=8)

## 6. Tổng hợp toàn bộ 4 mô hình

In [13]:
# Đọc từ đĩa để bảng vẫn đầy đủ khi Part A và Part B chạy ở hai Version khác nhau.
import glob

rows = []
for f in sorted(glob.glob(os.path.join(LOG_DIR, "xlmr_*_summary.json"))):
    with open(f, encoding="utf-8") as fh:
        rows.append(json.load(fh))
table = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
display(table.round(4))
table.to_csv(os.path.join(LOG_DIR, "xlmr_results.csv"), index=False)

frames = [table]
for prev in ("cnn_rnn_results.csv", "phobert_results.csv"):
    path = os.path.join(LOG_DIR, prev)
    if os.path.exists(path):
        frames.append(pd.read_csv(path))

if len(frames) > 1:
    allr = pd.concat(frames, ignore_index=True).sort_values("accuracy", ascending=False)
    display(allr[["run_name", "accuracy", "macro_f1", "params", "train_seconds"]].round(4))
    allr.to_csv(os.path.join(LOG_DIR, "all_results.csv"), index=False)
    print("Đã ghi outputs/logs/all_results.csv — dùng cho 05_analysis.ipynb")

,run_name,model_id,accuracy,macro_f1,weighted_f1,params,epochs_chạy,train_seconds,inference_ms_per_sample,peak_vram_gb
0,xlmr_large_finetuned,xlm-roberta-large,0.449,0.3721,0.3723,559893507,5,2807.2,16.23,13.47
1,xlmr_zeroshot,joeddav/xlm-roberta-large-xnli,0.334,0.3246,0.3246,559893507,0,0.0,4.65,NaN


In [14]:
# predictions/*.npy đã được save_run() ghi ngay sau từng run.
pred_dir = os.path.join(LOG_DIR, "predictions")
print(sorted(os.listdir(pred_dir)))

['xlmr_large_finetuned.npy', 'xlmr_zeroshot.npy', 'y_true.npy']


In [15]:
best = max((r for r in RESULTS.values()), key=lambda r: r["accuracy"])["run_name"]
y_true, y_pred = RESULTS[best]["_preds"]
print(f"Tốt nhất: {best}")
pd.set_option("display.max_colwidth", 100)
pd.DataFrame(error_examples(ds["test"], cols, y_true, y_pred, n=10))

Tốt nhất: xlmr_large_finetuned


,premise,hypothesis,gold,pred
0,"Sáng 23/5, ông Trần Văn Vịnh, Chủ tịch UBND phường Mai Động, cho biết lý do phong tỏa là nam sin...",Theo thông tin một công chức Nhà nước ở phường Mai Động đã có ít nhất 11 nam sinh đã bị mắc Covi...,contradiction,entailment
1,Cảnh sát thành phố Bhopal hôm 13/5 cho biết sự việc gây sốc xảy ra đầu tháng 4 và được công bố s...,Sự việc gây sốc xảy ra đầu tháng 4 và nghi phạm bị bắt sau nữa tháng.,contradiction,neutral
2,"Sau khi xem xét camera giám sát trong khu vực, cảnh sát nhận định nạn nhân đã bị cướp tài sản và...",Cảnh sát đã phán đoán bằng nhận định sắc bén sau khi xem camera an ninh.,neutral,entailment
3,"3h sáng cùng ngày, trái tim của của người hiến đã đập lại trong lồng ngực chàng trai.","3am, một người được tái sinh.",contradiction,entailment
4,"Ông Phạm Cao Vỹ, chủ tịch Hiệp hội Du lịch Sa Pa chia sẻ, nhận được lời kêu gọi từ hiệp hội, rất...",Ông Phạm Cao Vỹ được yêu thích.,neutral,entailment
5,"Ngành y tế Quảng Nam ngày 16/5 ghi nhận 3.070 mẫu xét nghiệm âm tính nCoV, 476 mẫu đang chờ kết ...",Giữa tháng 5 số lượng mẫu âm tính tại Quảng Nam rất cao và cao thứ 2 cả nước.,neutral,entailment
6,"Đạo diễn không đồng tình việc diễn viên bình thường, không mắc bệnh tật đi sửa mặt.",Đạo diễn không đồng tình việc vì theo đuổi cái đẹp mà đụng chạm dao kéo của diễn viên.,entailment,neutral
7,"Theo Vogue, đôi dép của Questlove mạ vàng 24 carat, giúp nhạc sĩ Mỹ tạo nét khác biệt.",Đôi dép của nhạc sĩ tên Mỹ được làm từ vàng 24 carat.,contradiction,neutral
8,"Đang nóng ruột vì chưa biết làm thế nào để bay về sớm với hai con gái bị tai nạn, anh Trần Văn H...",Anh Trần Văn Hải không có ý định hỏi vé từ một người lạ.,neutral,entailment
9,"Nguyễn Xuân Thành, 17 tuổi, thừa nhận có xích mích với cụ bà 88 tuổi sống gần nhà nên sát hại, t...",Cụ bà 88 tuổi đã tử vong.,entailment,neutral


## Ghi chú cho Kaggle

- XLM-R large ~560M tham số — đây là notebook tốn quota nhất. Chạy **Save & Run All**.
- Nếu OOM: `batch_size=4, grad_accum=8`, hoặc hạ `MAX_LENGTH` xuống 192.
- Nếu vẫn không đủ tài nguyên: dùng `xlm-roberta-base` và **ghi rõ lý do thay đổi** trong
  báo cáo (mục "Điều chỉnh của nhóm" trong bảng thông tin mô hình bên ngoài).
- Cân nhắc tách Phần A và Phần B thành hai Version để không mất kết quả zero-shot nếu
  phần fine-tune sập.

## Cần điền vào báo cáo

- **Bảng thông tin nguồn của mô hình bên ngoài** — chép từ đầu notebook này, nhớ điền commit hash.
- Bảng siêu tham số (cột External).
- Hình `xlmr_large_finetuned_curve.png`, các confusion matrix.
- Ba luận điểm để thảo luận:
  1. **Zero-shot XNLI vs fine-tune ViANLI** — kiến thức NLI đa ngữ chuyển giao được bao nhiêu?
  2. **XLM-R large vs PhoBERT-base** — mô hình lớn hơn/đa ngữ có thắng mô hình chuyên tiếng Việt không, và đổi lại bao nhiêu chi phí tính toán?
  3. **`xlmr_xnli_finetuned` vs `xlmr_large_finetuned`** — pretrain trung gian trên XNLI có giúp ích?